# v3: Element-wise Operations

The important thing to realize is that when we write
```py
c = a + b
```

Python actually calls
```py
c = a.__add__(b)
```

Similarly,
```py
a - b   --> a.__sub__(b)
a * b   --> a.__mul__(b)
a / b   --> a.__truediv__(b)
```

So we'll implement those methods directly.

In [ ]:
class ndarray(ndarray):
    def __add__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a + b)

        else:  # scalar
            for a in self.data:
                result.append(a + other)

        return ndarray(result)

    def __sub__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a - b)

        else:
            for a in self.data:
                result.append(a - other)

        return ndarray(result)

    def __mul__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a * b)

        else:
            for a in self.data:
                result.append(a * other)

        return ndarray(result)

    def __truediv__(self, other):
        result = []

        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            for a, b in zip(self.data, other.data):
                result.append(a / b)

        else:
            for a in self.data:
                result.append(a / other)

        return ndarray(result)


def array(data):
    return ndarray(data)

In [ ]:
a = array([1, 2, 3])
b = array([4, 5, 6])

print(a + b)
# array([5, 7, 9])

print(a - b)
# array([-3, -3, -3])

print(a * b)
# array([4, 10, 18])

print(b / a)
# array([4.0, 2.5, 2.0])

print(a + 10)
# array([11, 12, 13])

print(a * 5)
# array([5, 10, 15])

array([5, 7, 9])
array([-3, -3, -3])
array([4, 10, 18])
array([4.0, 2.5, 2.0])
array([11, 12, 13])
array([5, 10, 15])


**A small observation**

You probably noticed that __add__, __sub__, __mul__, and __truediv__ are almost identical. Real libraries avoid this repetition by having all four call a common internal helper. For learning purposes, though, writing them out separately makes it much easier to see exactly what each operator is doing before we start refactoring.

# v4: Reduction Methods (Exercise)

In [ ]:
a = np.array([1, 2, 3, 4])

print(a.sum())     # 10
print(a.mean())    # 2.5
print(a.min())     # 1
print(a.max())     # 4

10
2.5
1
4


Home work:
- `prod()`
- `std()`
- `var()`
- `argmin()`
- `argmax()`

**What you just learned**

Until now, every operation returned another array.

a + b --> array([...])

A reduction is different.

array([5, 1, 8, 2]) --->reduce(sum)---> 16

Instead of producing another array, it produces a single number.

# v5: 2D (Part I)

Until now, our array has only supported 1D data.

But NumPy is designed for N-dimensional arrays.

Let's first support 2D arrays.

Currently we have
```py
self.shape = (len(data),)
```

Instead, detect whether data is 1D or 2D.

In [ ]:
class ndarray(ndarray):
    def __init__(self, data):
        self.data = data
        if isinstance(data[0], list):
            row = len(data)
            col = len(data[0])
            self.shape = (row, col)
        else:
            self.shape = (len(data),)

a = array([[1,2,3],[4,5,6]])
a.shape

(2, 3)

**Validate the input**

What if someone writes

```py
array([
    [1,2],
    [3,4,5]
])
```

Is this a valid matrix?

In [ ]:
np.array([[1,2,9],[3,4,5,],[3,4,5,0]])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

Every row must have the same length.

Add this check:

In [ ]:
class ndarray(ndarray):
    def __init__(self, data):
        self.data = data
        if isinstance(data[0], list):
            len_row = len(data)
            len_col = len(data[0])
            for row in data:
                if len(row) != len_col:
                    raise ValueError(f"setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was ({row},) + inhomogeneous part.")

            self.shape = (len_row, len_col)
        else:
            self.shape = (len(data),)

a = array([[1,2,3],[4,5]])

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was ([4, 5],) + inhomogeneous part.

Now we have an issue with indexing:

In [ ]:
a = np.array([
    [1,2,3],
    [4,5,6]
])
a[0]

array([1, 2, 3])

But in our module:

In [ ]:
a = array([
    [1,2,3],
    [4,5,6]
])
a[0]

[1, 2, 3]

It doesn't return a ndarray. It returns a regular python list

So, let's fix it

In [ ]:
class ndarray(ndarray):
    def __getitem__(self, index):
        value = self.data[index]
        if isinstance(value, list):
            return ndarray(value)
        return value

# now let's try it, again
a = array([
    [1,2,3],
    [4,5,6]
])
a[0]

array([1, 2, 3])

But there's a problem...

Our addition code assumes the data is flat.

For example,

In [ ]:
a = array([
    [1,2],
    [3,4]
])

b = array([
    [5,6],
    [7,8]
])

a + b

array([[1, 2, 5, 6], [3, 4, 7, 8]])

Our current loop does

```py
for a, b in zip(self.data, other.data):
    result.append(a + b)
```

But here `a` is actually `[1,2]` and `b` is `[5,6]`

Python list addition means concatenation, so you'd get:

[1,2] + [5,6] = [1,2,5,6]

instead of

[1,2] + [5,6] = [6,8]

So the next stage is where we'll make our arithmetic truly N-dimensional by teaching it to recursively apply operations to nested lists. That will let the same +, -, *, and / code work for both 1D and 2D arrays, bringing our implementation much closer to how NumPy thinks about arrays.

# v6: 2D (Part II)

This is where our implementation starts to resemble the real design of NumPy.

Instead of writing separate code for 1D arrays and 2D arrays, we'll think recursively.

**The idea**

Consider these two arrays:

```py
a = [
    [1, 2],
    [3, 4]
]

b = [
    [5, 6],
    [7, 8]
]
```

What is a + b?

Don't think about the whole matrix at once. Think recursively.
```py
[
    [1,2] + [5,6],
    [3,4] + [7,8]
]
```

Now break it down again.
```py
[
    [
        1+5,
        2+6
    ],
    [
        3+7,
        4+8
    ]
]
```

Eventually you reach numbers, and numbers know how to add themselves.

This suggests a recursive algorithm.

First, let's add a recursive helper

Add this method to your class.

In [ ]:
class ndarray(ndarray):
    def _apply_op(self, left, right, op):
        if not isinstance(left, list):
            return op(left, right)

        result = []

        for a, b in zip(left, right):
            result.append(self._apply_op(a, b, op))

        return result

Notice that this method doesn't know whether it's working on a vector, matrix, or higher-dimensional tensor.

It simply asks:

"Am I looking at numbers?"

Yes --> perform the operation.
No --> go one level deeper.

Next, we rewrite __add__

Now __add__ becomes much shorter.

In [ ]:
class ndarray(ndarray):
    def __add__(self, other):
        if isinstance(other, ndarray):
            if self.shape != other.shape:
                raise ValueError("Shapes must match.")

            return ndarray(
                self._apply_op(
                    self.data,
                    other.data,
                    lambda a, b: a + b
                )
            )

        else:
            raise NotImplementedError

Test it

In [ ]:
# 1D
a = array([1,2,3])
b = array([4,5,6])

print(a + b)
# array([5,7,9])

# 2D
a = array([
    [1,2],
    [3,4]
])

b = array([
    [5,6],
    [7,8]
])

print(a + b)
# array([
#     [6,8],
#     [10,12]
# ])

array([5, 7, 9])
array([[6, 8], [10, 12]])


**Why recursion?**

Imagine we later support a 3D tensor.

```py
[
    [
        [1,2],
        [3,4]
    ],
    [
        [5,6],
        [7,8]
    ]
]
```

Our algorithm still works.

It keeps descending until it reaches numbers.

That's one of the beautiful ideas behind NumPy: operations are defined over arbitrary dimensions, not just vectors or matrices.

**Exercise:** Implement `__mul__`, `__sub__`, `__truediv__`

Our implementation still has a major limitation:

```py
a = array([1,2,3])

a + 10
```

doesn't work anymore because `_apply_op()` expects both operands to have the same nested structure.

The next feature we'll implement is scalar broadcasting. Once that's done, expressions like

```py
a + 10
a * 5

matrix + 100
matrix * 2
```

will all work naturally, and we'll be one step closer to real NumPy's broadcasting model.

# v7: Tuple Indexing
(Exercise)

# v8: Broadcasting

Try it yourself.

Don't implement with AI, because that way, you won't learn anything.

# v9: Transpose

# v10: Matrix Multiplication